In [ ]:
import requests
# !pip install yfinance
import yfinance as yf
import pandas as pd
from io import StringIO
from pathlib import Path
from collections import defaultdict

In [ ]:
def get_earnings_yahoo_finance(symbol):
    """
    Alternative using yfinance library
    Install with: pip install yfinance
    """
    try:


        ticker = yf.Ticker(symbol)



        # Get earnings data
        earnings = ticker.get_earnings_dates(limit=20)

        if earnings is not None and not earnings.empty:
            # Reset index to make date a column
            df = earnings.reset_index()

            # Rename columns
            df = df.rename(columns={
                'Earnings Date': 'Date Reported',
                'EPS Estimate': 'Consensus EPS Forecast',
                'Reported EPS': 'Earnings Per Share'
            })

            # Calculate surprise percentage
            if 'Earnings Per Share' in df.columns and 'Consensus EPS Forecast' in df.columns:
                df['% Surprise'] = ((df['Earnings Per Share'] - df['Consensus EPS Forecast']) /
                                  df['Consensus EPS Forecast'] * 100).round(2)

            result = df[df['Event Type'] == 'Earnings']

            return result
        else:
            print("No earnings data found")
            return None

    except ImportError:
        print("yfinance not installed. Install with: pip install yfinance")
        return None
    except Exception as e:
        print(f"Error fetching Yahoo Finance data: {e}")
        return None


In [ ]:
# scraping Wikipedia site
r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={'User-agent': 'Chrome/103.0.5060.114'}).text
df = pd.read_html(StringIO(str(r))) #load with user agent to avoid 401 error
CIK_symbols = pd.DataFrame(df[0])
CIK_symbols['CIK'] = CIK_symbols['CIK'].astype(str).str.zfill(10)

In [ ]:
# CIK_symbols = CIK_symbols[-40:]

In [ ]:
print(CIK_symbols)

    Symbol             Security             GICS Sector  \
0      MMM                   3M             Industrials   
1      AOS          A. O. Smith             Industrials   
2      ABT  Abbott Laboratories             Health Care   
3     ABBV               AbbVie             Health Care   
4      ACN            Accenture  Information Technology   
..     ...                  ...                     ...   
498    XYL           Xylem Inc.             Industrials   
499    YUM          Yum! Brands  Consumer Discretionary   
500   ZBRA   Zebra Technologies  Information Technology   
501    ZBH        Zimmer Biomet             Health Care   
502    ZTS               Zoetis             Health Care   

                                GICS Sub-Industry    Headquarters Location  \
0                        Industrial Conglomerates    Saint Paul, Minnesota   
1                               Building Products     Milwaukee, Wisconsin   
2                           Health Care Equipment  North 

In [ ]:
out_file = Path("earnings_data_yfin.xlsx")
first_write = True
for index, each_row in CIK_symbols.iterrows():
  count = 0
  # print(f"\n{each_row['Symbol']}:")
  sheet_name = str(each_row['Symbol'])
  try:
    if not '.' in each_row['Symbol']:
      symbol_send_to_api = each_row['Symbol']
    else: symbol_send_to_api = str(each_row['Symbol']).replace('.','-')
    earnings_df = get_earnings_yahoo_finance(symbol_send_to_api)
    earnings_df['Date Reported'] = earnings_df['Date Reported'].dt.tz_localize(None)
    earnings_df['Date'] = pd.to_datetime(earnings_df['Date Reported']).dt.date
    earnings_df['Time'] = pd.to_datetime(earnings_df['Date Reported']).dt.time
    # Below logic of count was written as, for three companies TICKR symbols - yfinance was returning for all 4 quarters as sec filings.
    # with 0 count: Ignore. nothing availabel on yfinance. Checked.
    # with count 1: CTAS, sec also gave only two. CAG: sec also gave only two. LW: sec also gave only two.
    # with count 3: STZ, sec gave 3. MKC, sec gave 3. WBA, sec gave 3.
    # Ruke used:
    # Rule used by me:
    # with count 2, remove three rows. Normal.
    # with count 3, remove four rows.
    # with count 1, remove three rows. Normal.
    # with count 0, ignore.
    for _, each_date in earnings_df.iterrows():
      if '2025' in str(each_date['Date']): count += 1
      else: continue
    if count in (1, 2): earnings_df = earnings_df[3:]
    elif count == 3: earnings_df = earnings_df[4:]
    else: earnings_df = earnings_df # when count = 0, company not found on yfinance
    # print(count)
    mode = "w" if first_write else "a"
    if mode == "w":
        with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode) as writer:
          earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
        with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode, if_sheet_exists="new") as writer:
          earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)
    first_write = False
  except Exception as e:
    print(f"Error for symbol {each_row['Symbol']}: {e}")

ERROR:yfinance:PSKY: $PSKY: possibly delisted; no earnings dates found


No earnings data found
Error for symbol PSKY: 'NoneType' object is not subscriptable


In [ ]:
# # was checking if count logic is working. Please ignore.
# data_dict = defaultdict(int)
# for index, each_row in CIK_symbols.iterrows():
#   count = 0
#   print(f"\n{each_row['Symbol']}:")
#   try:
#     if not '.' in each_row['Symbol']:
#       symbol_send_to_api = each_row['Symbol']
#     else: symbol_send_to_api = str(each_row['Symbol']).replace('.','-')
#     earnings_df = get_earnings_yahoo_finance(symbol_send_to_api)
#     earnings_df['Date Reported'] = earnings_df['Date Reported'].dt.tz_localize(None)
#     earnings_df['Date'] = pd.to_datetime(earnings_df['Date Reported']).dt.date
#     earnings_df['Time'] = pd.to_datetime(earnings_df['Date Reported']).dt.time
#     for _, each_date in earnings_df.iterrows():
#       if '2025' in str(each_date['Date']): count += 1
#       else: continue
#     data_dict[each_row['Symbol']] = count
#   except Exception as e:
#     print(f"Error for symbol {each_row['Symbol']}: {e}")
#     data_dict[each_row['Symbol']] = 0

# df = pd.DataFrame(list(data_dict.items()), columns=['SYMBOL', 'count'])
# print(df)
# df.to_excel('count.xlsx', index=False)


MMM:

AOS:

ABT:

ABBV:

ACN:

ADBE:

AMD:

AES:

AFL:

A:

APD:

ABNB:

AKAM:

ALB:

ARE:

ALGN:

ALLE:

LNT:

ALL:

GOOGL:

GOOG:

MO:

AMZN:

AMCR:

AEE:

AEP:

AXP:

AIG:

AMT:

AWK:

AMP:

AME:

AMGN:

APH:

ADI:

AON:

APA:

APO:

AAPL:

AMAT:

APTV:

ACGL:

ADM:

ANET:

AJG:

AIZ:

T:

ATO:

ADSK:

ADP:

AZO:

AVB:

AVY:

AXON:

BKR:

BALL:

BAC:

BAX:

BDX:

BRK.B:

BBY:

TECH:

BIIB:

BLK:

BX:

XYZ:

BK:

BA:

BKNG:

BSX:

BMY:

AVGO:

BR:

BRO:

BF.B:

BLDR:

BG:

BXP:

CHRW:

CDNS:

CZR:

CPT:

CPB:

COF:

CAH:

KMX:

CCL:

CARR:

CAT:

CBOE:

CBRE:

CDW:

COR:

CNC:

CNP:

CF:

CRL:

SCHW:

CHTR:

CVX:

CMG:

CB:

CHD:

CI:

CINF:

CTAS:

CSCO:

C:

CFG:

CLX:

CME:

CMS:

KO:

CTSH:

COIN:

CL:

CMCSA:

CAG:

COP:

ED:

STZ:

CEG:

COO:

CPRT:

GLW:

CPAY:

CTVA:

CSGP:

COST:

CTRA:

CRWD:

CCI:

CSX:

CMI:

CVS:

DHR:

DRI:

DDOG:

DVA:

DAY:

DECK:

DE:

DELL:

DAL:

DVN:

DXCM:

FANG:

DLR:

DG:

DLTR:

D:

DPZ:

DASH:

DOV:

DOW:

DHI:

DTE:

DUK:

DD:

EMN:

ETN:



ERROR:yfinance:PSKY: $PSKY: possibly delisted; no earnings dates found


No earnings data found
Error for symbol PSKY: 'NoneType' object is not subscriptable

PH:

PAYX:

PAYC:

PYPL:

PNR:

PEP:

PFE:

PCG:

PM:

PSX:

PNW:

PNC:

POOL:

PPG:

PPL:

PFG:

PG:

PGR:

PLD:

PRU:

PEG:

PTC:

PSA:

PHM:

PWR:

QCOM:

DGX:

RL:

RJF:

RTX:

O:

REG:

REGN:

RF:

RSG:

RMD:

RVTY:

ROK:

ROL:

ROP:

ROST:

RCL:

SPGI:

CRM:

SBAC:

SLB:

STX:

SRE:

NOW:

SHW:

SPG:

SWKS:

SJM:

SW:

SNA:

SOLV:

SO:

LUV:

SWK:

SBUX:

STT:

STLD:

STE:

SYK:

SMCI:

SYF:

SNPS:

SYY:

TMUS:

TROW:

TTWO:

TPR:

TRGP:

TGT:

TEL:

TDY:

TER:

TSLA:

TXN:

TPL:

TXT:

TMO:

TJX:

TKO:

TTD:

TSCO:

TT:

TDG:

TRV:

TRMB:

TFC:

TYL:

TSN:

USB:

UBER:

UDR:

ULTA:

UNP:

UAL:

UPS:

URI:

UNH:

UHS:

VLO:

VTR:

VLTO:

VRSN:

VRSK:

VZ:

VRTX:

VTRS:

VICI:

V:

VST:

VMC:

WRB:

GWW:

WAB:

WBA:

WMT:

DIS:

WBD:

WM:

WAT:

WEC:

WFC:

WELL:

WST:

WDC:

WY:

WSM:

WMB:

WTW:

WDAY:

WYNN:

XEL:

XYL:

YUM:

ZBRA:

ZBH:

ZTS:
    SYMBOL  count
0      MMM      2
1      AOS   

In [ ]:
get_earnings_yahoo_finance("FOX")

,Date Reported,Consensus EPS Forecast,Earnings Per Share,Surprise(%),Event Type,% Surprise


In [ ]:
multiple_tickers = [ "XOM", "AAPL", "MSFT", "GOOG", "AMZN", "TSLA", "NVDA", "META", "NFLX", "CSCO", "INTC", "IBM", "ORCL", "SAP"]

tickers_with_earnings = {}


for i in multiple_tickers:
    earnings_data = get_earnings_yahoo_finance(i)
    if earnings_data is not None:
        tickers_with_earnings[i] = earnings_data
    else:
        print(f"No earnings data found for {i}")


In [ ]:
tickers_with_earnings['FOX']

NameError: name 'tickers_with_earnings' is not defined

In [ ]:
tickers = yf.Ticker('SPY').funds_data
tickers.description

'The trust seeks to achieve its investment objective by holding a portfolio of the common stocks that are included in the index, with the weight of each stock in the portfolio substantially corresponding to the weight of such stock in the index.'

In [ ]:
tickers.top_holdings

,Name,Holding Percent
Symbol,,
NVDA,NVIDIA Corp,0.073195
MSFT,Microsoft Corp,0.070233
AAPL,Apple Inc,0.058215
AMZN,Amazon.com Inc,0.039380
META,Meta Platforms Inc Class A,0.030441
AVGO,Broadcom Inc,0.024621
GOOGL,Alphabet Inc Class A,0.019483
BRK-B,Berkshire Hathaway Inc Class B,0.016921
TSLA,Tesla Inc,0.016909


In [ ]:
company_list = pd.read_html('Table.html')[0]
tickers_with_earnings = {}

for i in company_list['Symbol'].tolist():
    earnings_data = get_earnings_yahoo_finance(i)
    if earnings_data is not None:
        tickers_with_earnings[i] = earnings_data
    else:
        print(f"No earnings data found for {i}")



BRK.B: $BRK.B: possibly delisted; no earnings dates found


No earnings data found
No earnings data found for BRK.B


BF.B: $BF.B: possibly delisted; no earnings dates found


No earnings data found
No earnings data found for BF.B


In [ ]:
# Save the ticker_with_earnings into a json file (write the code to do this)












ValueError: If using all scalar values, you must pass an index